# Options Pricing from a Trained MPS Born Machine

Loads a saved MPS bundle, draws Monte Carlo paths by sequential conditional
sampling, and prices four exotic option types:

| Option | Payoff |
|---|---|
| European call | max(S_T − K, 0) |
| Asian call | max(mean(Sₜ) − K, 0) |
| Lookback call | max(max(Sₜ) − K, 0) |
| Up-and-out barrier call | max(S_T − K, 0) · 𝟙[max(Sₜ) < B] |

For the European option the Black-Scholes implied volatility is also computed,
and the implied-vol smile is plotted across a range of strikes.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "../.."))
Pkg.resolve()
Pkg.instantiate()

using MPSFast
using MPSFast.Encoders
using LinearAlgebra, Statistics, Printf, Plots
using Distributions: Normal, cdf

  Activating project at `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl`
     Project No packages added to or removed from `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/Project.toml`
    Manifest No packages added to or removed from `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/Manifest.toml`


## 1. Configuration

Set the path to a saved MPS bundle and the encoder parameters used when
training it.  The meta dict in the bundle stores `D_max`, `M`, and `m`;
you only need to set `encoder_type` and the grid limits (`S_min`, `S_max`)
which are not persisted automatically.

In [2]:
# ── Path to saved model ────────────────────────────────────────────────────────
mps_file = "mps_heston.jld2"   # adjust to your saved bundle

# ── Encoder type: BasisEncoder, BinaryEncoder, or TrigEncoder ─────────────────
encoder_type = BasisEncoder

# ── Simulation parameters ─────────────────────────────────────────────────────
N_samp   = 10_000   # number of MPS-sampled paths
S0       = 100.0    # spot price (used for option pricing and BS reference)
r        = 0.0      # risk-free rate

# ── Option parameters ─────────────────────────────────────────────────────────
K_base   = 100.0    # ATM strike for the comparison table
B_level  = 115.0    # barrier level for the up-and-out option

# ── IV smile: strikes as fraction of S0 ───────────────────────────────────────
smile_moneyness = range(0.75, 1.30; length = 30)

0.75:0.01896551724137931:1.3

## 2. Load MPS and Reconstruct Encoder

In [3]:
mps, nll_hist, epoch_saved, meta = load_mps_bundle(mps_file)

M     = meta["M"]     # chain length (time steps)
m     = meta["m"]     # encoder bits / resolution parameter
D_max = meta["D_max"]
T_mat = M / 252.0     # maturity in years (assuming daily steps)

@printf("Loaded MPS: M=%d  m=%d  D_max=%d  epoch=%d  NLL=%.4f\n",
        M, m, D_max, epoch_saved, nll_hist[end])
println("Bond dims: ", [size(mps[j], 3) for j in 1:M-1])

# Reconstruct encoder from meta.
# If the bundle was saved BEFORE the Smin/Smax fix, fall back to manual override.
enc_name = get(meta, "encoder", string(encoder_type))
enc = encoder_type(m)
if haskey(meta, "Smin") && haskey(meta, "Smax")
    enc.Smin = Float64(meta["Smin"])
    enc.Smax = Float64(meta["Smax"])
    @printf("Encoder: %s  grid=[%.2f, %.2f]  d=%d buckets\n",
            enc_name, enc.Smin, enc.Smax, site_dim(enc))
else
    # ── MANUAL FALLBACK ────────────────────────────────────────────────────
    # Bundle was saved without encoder grid info.  Set Smin/Smax here to
    # match the training data range used when the model was trained.
    # Example: enc.Smin = 60.0; enc.Smax = 160.0
    # Alternatively, reload training paths and call fit_grid!(enc, paths).
    @warn "Smin/Smax not in meta. Set enc.Smin and enc.Smax manually in the next cell."
end

Loaded MPS: M=30  m=4  D_max=150  epoch=50  NLL=16.2194
Bond dims: [16, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 16]


┌ Warning: Smin/Smax not in meta. Set enc.Smin and enc.Smax manually in the next cell.
└ @ Main /Users/bi006881/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/notebooks/experiments/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X31sZmlsZQ==.jl:27


## 3. Sample Paths

In [ ]:
sampled_paths, sampled_xi = sample_paths(enc, mps, N_samp; seed = 42)

@printf("Sampled %d paths of length %d\n", size(sampled_paths)...)
@printf("Price range: [%.2f, %.2f]\n", minimum(sampled_paths), maximum(sampled_paths))

p_paths = plot(sampled_paths[1:100, :]';
    alpha = 0.2, lw = 0.8, color = :steelblue, legend = false,
    xlabel = "time step", ylabel = "price",
    title  = "100 MPS-sampled paths", size = (700, 300))
display(p_paths)

## 4. Pricing Functions

In [ ]:
"""Price four option types on a matrix of paths (N × M).  Returns a NamedTuple."""
function price_options(paths::AbstractMatrix, K, B; r = 0.0, T = 1.0)
    disc  = exp(-r * T)
    ST    = paths[:, end]
    Savg  = vec(mean(paths; dims = 2))
    Smax  = vec(maximum(paths; dims = 2))
    (
        european = disc * mean(max.(ST  .- K, 0.0)),
        asian    = disc * mean(max.(Savg .- K, 0.0)),
        lookback = disc * mean(max.(Smax .- K, 0.0)),
        barrier  = disc * mean(max.(ST  .- K, 0.0) .* (Smax .< B)),
    )
end

"""Black-Scholes European call price."""
function bs_call(S, K, T, σ; r = 0.0)
    d1 = (log(S / K) + (r + σ^2 / 2) * T) / (σ * sqrt(T))
    d2 = d1 - σ * sqrt(T)
    S * cdf(Normal(), d1) - K * exp(-r * T) * cdf(Normal(), d2)
end

"""Newton–bisection implied vol from a call price."""
function implied_vol(price, S, K, T; r = 0.0, tol = 1e-8, max_iter = 200)
    f(σ) = bs_call(S, K, T, σ; r = r) - price
    lo, hi = 1e-5, 10.0
    f(lo) * f(hi) > 0 && return NaN
    for _ in 1:max_iter
        mid = (lo + hi) / 2
        abs(f(mid)) < tol && return mid
        f(lo) * f(mid) < 0 ? (hi = mid) : (lo = mid)
    end
    return (lo + hi) / 2
end

## 5. Comparison Table (ATM, K = K_base)

In [ ]:
pm = price_options(sampled_paths, K_base, B_level; r = r, T = T_mat)

# Black-Scholes European reference at σ = 20%
bs_ref  = bs_call(S0, K_base, T_mat, 0.20; r = r)
iv_mps  = implied_vol(pm.european, S0, K_base, T_mat; r = r)

println("Option prices  (K = $(K_base),  B = $(B_level),  T = $(round(T_mat; digits=3)) yr)")
println("─"^62)
@printf("%-20s  %10s  %10s  %10s\n", "Type", "MPS MC", "BS (σ=20%)", "Diff")
println("─"^62)
@printf("%-20s  %10.4f  %10.4f  %+10.4f\n",
        "European call",  pm.european, bs_ref, pm.european - bs_ref)
@printf("%-20s  %10.4f  %10s  %10s\n",
        "Asian call",     pm.asian,    "—", "—")
@printf("%-20s  %10.4f  %10s  %10s\n",
        "Lookback call",  pm.lookback, "—", "—")
@printf("%-20s  %10.4f  %10s  %10s\n",
        "Barrier (U&O)",  pm.barrier,  "—", "—")
println("─"^62)
@printf("Implied vol (European):  %.2f%%\n", iv_mps * 100)

## 6. Option Prices Across Strikes

Sweep strikes from 75 % to 130 % of spot and price each option type.

In [ ]:
strikes = S0 .* smile_moneyness

eu_prices = [price_options(sampled_paths, K, B_level; r=r, T=T_mat).european for K in strikes]
as_prices = [price_options(sampled_paths, K, B_level; r=r, T=T_mat).asian    for K in strikes]
lb_prices = [price_options(sampled_paths, K, B_level; r=r, T=T_mat).lookback for K in strikes]
ba_prices = [price_options(sampled_paths, K, B_level; r=r, T=T_mat).barrier  for K in strikes]

p_prices = plot(strikes, eu_prices; lw=2, label="European",  color=:steelblue)
plot!(p_prices, strikes, as_prices; lw=2, label="Asian",     color=:darkorange)
plot!(p_prices, strikes, lb_prices; lw=2, label="Lookback",  color=:forestgreen)
plot!(p_prices, strikes, ba_prices; lw=2, label="Barrier U&O", color=:crimson)
vline!(p_prices, [S0]; ls=:dash, color=:gray, lw=1, label="ATM")
plot!(p_prices;
    xlabel = "strike K", ylabel = "option price",
    title  = "MPS option prices vs. strike",
    legend = :topright, size = (700, 350))
display(p_prices)

## 7. Implied-Volatility Smile

Invert Black-Scholes for the European call at each strike.  Deviations from
the flat 20 % line reveal skewness or fat-tail structure learned by the MPS.

In [ ]:
ivols = [implied_vol(p, S0, K, T_mat; r = r) * 100
         for (p, K) in zip(eu_prices, strikes)]

p_smile = plot(strikes, ivols;
    lw=2, marker=:circle, markersize=3, color=:darkorange,
    label = "MPS implied vol",
    xlabel = "strike K", ylabel = "implied vol (%)",
    title  = "Implied-vol smile — MPS vs flat 20%",
    size = (700, 320), legend = :top)
hline!(p_smile, [20.0]; lw=1.5, ls=:dash, color=:gray, label="20% flat")
vline!(p_smile, [S0];   lw=1,   ls=:dot,  color=:black, label="ATM")
display(p_smile)

## 8. NLL Training History

In [ ]:
plot(nll_hist;
    lw=2, marker=:circle, markersize=3, color=:steelblue,
    xlabel = "epoch", ylabel = "NLL",
    title  = "Training NLL history",
    legend = false, size = (600, 300))